In [1]:
import numpy as np
from collections import defaultdict

class NaiveBayes:
    def __init__(self):
        self.class_probs = {}        # P(class)
        self.feature_probs = {}      # P(feature|class)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        self.class_probs = {}
        self.feature_probs = {}

        # Prior probability of each class
        for cls in self.classes:
            X_cls = X[y == cls]
            self.class_probs[cls] = len(X_cls) / n_samples
            self.feature_probs[cls] = {
                "mean": X_cls.mean(axis=0),
                "var": X_cls.var(axis=0) + 1e-6  # Add small value to avoid zero
            }

    def _gaussian_pdf(self, x, mean, var):
        # Gaussian probability density function
        exponent = np.exp(- ((x - mean) ** 2) / (2 * var))
        return (1 / np.sqrt(2 * np.pi * var)) * exponent

    def _predict_single(self, x):
        posteriors = []

        for cls in self.classes:
            prior = np.log(self.class_probs[cls])  # log to avoid underflow
            class_conditional = np.sum(np.log(
                self._gaussian_pdf(x, self.feature_probs[cls]["mean"], self.feature_probs[cls]["var"])
            ))
            posterior = prior + class_conditional
            posteriors.append(posterior)

        return self.classes[np.argmax(posteriors)]

    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load dataset
data = load_iris()
X, y = data.data, data.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train model
model = NaiveBayes()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
accuracy = np.mean(y_pred == y_test)
print("Accuracy:", accuracy)


Accuracy: 0.9333333333333333
